# 06 Prepare stocks

This notebook prepares the yearly **stocks / market view** used later in the thesis.

Purpose:
- reshape monthly stock-related source tables
- create yearly market signals
- keep the output focused on reusable yearly view construction
- avoid heavy downstream feature engineering inside the preparation step

The output from this notebook is a reusable `stock_view.csv` table that can later be:
- explored in `13_stocks_eda.ipynb`
- joined into the contract master table
- used for Snorkel, feature engineering, and modeling

## Imports and project path setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

project_root: /Users/Thomas/Desktop/Master Thesis
src_path: /Users/Thomas/Desktop/Master Thesis/src


In [2]:
import re
import numpy as np
import pandas as pd

## Input and output paths

In [3]:
data_raw = project_root / "Data" / "raw"
data_stocks = project_root / "Data" / "Stocks"
data_processed = project_root / "Data" / "processed"

monthly_trading_path = data_stocks / "Stock_Monthly_Trading.csv"
monthly_market_cap_path = data_stocks / "Stock_Monthly_Market_Cap.csv"
beta_path = data_stocks / "Stocks_beta_correl_etc.csv"
monthly_shares_path = data_stocks / "Stocks_Monthly_Shares.csv"
monthly_closing_price_path = data_stocks / "Stocks_Monthly_ClosingPrices.csv"

data_processed.mkdir(parents=True, exist_ok=True)
output_path = data_processed / "stock_view.csv"

print(monthly_trading_path)
print(monthly_market_cap_path)
print(beta_path)
print(monthly_shares_path)
print(monthly_closing_price_path)
print(output_path)

/Users/Thomas/Desktop/Master Thesis/Data/Stocks/Stock_Monthly_Trading.csv
/Users/Thomas/Desktop/Master Thesis/Data/Stocks/Stock_Monthly_Market_Cap.csv
/Users/Thomas/Desktop/Master Thesis/Data/Stocks/Stocks_beta_correl_etc.csv
/Users/Thomas/Desktop/Master Thesis/Data/Stocks/Stocks_Monthly_Shares.csv
/Users/Thomas/Desktop/Master Thesis/Data/Stocks/Stocks_Monthly_ClosingPrices.csv
/Users/Thomas/Desktop/Master Thesis/Data/processed/stock_view.csv


## Helper functions

In [4]:
def clean_id_like(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

def parse_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def parse_numeric_with_comma_decimal(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

def parse_numeric_with_thousands_dot(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(".", "", regex=False),
        errors="coerce"
    )

def relative_trend(series: pd.Series) -> float:
    valid = series.dropna()
    if len(valid) < 2:
        return np.nan

    y = valid.values
    x = np.arange(len(y))
    slope = np.polyfit(x, y, 1)[0]

    mean_y = valid.mean()
    if mean_y == 0:
        return 0.0

    return slope / mean_y

def melt_monthly_source(
    df_source: pd.DataFrame,
    id_vars: list[str],
    pattern: str,
    var_name: str = "raw_header",
    value_name: str = "value",
) -> pd.DataFrame:
    compiled = re.compile(pattern)
    value_cols = [c for c in df_source.columns if compiled.search(c)]

    df_long = pd.melt(
        df_source,
        id_vars=id_vars,
        value_vars=value_cols,
        var_name=var_name,
        value_name=value_name,
    )

    extracted = df_long[var_name].str.extract(pattern)
    df_long["month"] = extracted[0]
    df_long["year"] = pd.to_numeric(extracted[1], errors="coerce").astype("Int64")

    return df_long

def add_join_year(df_input: pd.DataFrame, year_col: str = "year") -> pd.DataFrame:
    df_input = df_input.copy()
    df_input["Join_Year"] = pd.to_numeric(df_input[year_col], errors="coerce").astype("Int64") + 1
    return df_input

## 1. Monthly trading volume → yearly volume signals

In [5]:
df_trading_raw = pd.read_csv(
    monthly_trading_path,
    sep=";",
    engine="python",
    on_bad_lines="skip"
)

print("Shape:", df_trading_raw.shape)
display(df_trading_raw.head())

Shape: (1894, 387)


,Company name Latin alphabet,Risk level,BvD ID number,Monthly - Trading volume - January\nLast avail. yr,Monthly - Trading volume - January\nYear - 1,Monthly - Trading volume - January\nYear - 2,Monthly - Trading volume - January\nYear - 3,Monthly - Trading volume - January\nYear - 4,Monthly - Trading volume - January\nYear - 5,Monthly - Trading volume - January\nYear - 6,...,Monthly - Trading volume - December\n2020,Monthly - Trading volume - December\n2019,Monthly - Trading volume - December\n2018,Monthly - Trading volume - December\n2017,Monthly - Trading volume - December\n2016,Monthly - Trading volume - December\n2015,Monthly - Trading volume - December\n2014,Monthly - Trading volume - December\n2013,Monthly - Trading volume - December\n2012,Monthly - Trading volume - December\n2011
0,Jeev Lifeworks LLP,Do not source,IN0038742908,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,MY422033-W,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
2,Daicel Corporation,Go ahead,JP4120001125937,15.019.500,10.850.600,13.999.200,11.052.500,15.059.900,23.681.800,25.930.500,...,40.412.300,25.980.600,29.652.300,28.004.600,37.110.000,26.579.500,37.168.000,46.632.000,27.508.000,24.741.000
3,"Polyplastics Co.,Ltd.",Take caution,JP1010401053834,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
4,"Kolon Industries, Inc.",Go ahead,KR1353110013606,7.212.333,1.152.043,1.931.919,1.761.877,3.618.413,8.216.717,1.760.469,...,4.292.668,1.368.647,2.149.371,4.219.441,2.560.273,1.853.546,2.053.743,997.485,1.027.027,3.550.786


In [6]:
trading_pattern = r"Monthly - Trading volume - (\w+)\n(20\d{2})"

df_trading_long = melt_monthly_source(
    df_source=df_trading_raw,
    id_vars=["BvD ID number", "Company name Latin alphabet"],
    pattern=trading_pattern,
    var_name="raw_header",
    value_name="volume",
)

df_trading_long["BvD ID number"] = clean_id_like(df_trading_long["BvD ID number"])
df_trading_long["volume"] = parse_numeric(df_trading_long["volume"])

print("Long shape:", df_trading_long.shape)
display(df_trading_long.head())

Long shape: (363648, 6)


,BvD ID number,Company name Latin alphabet,raw_header,volume,month,year
0,IN0038742908,Jeev Lifeworks LLP,Monthly - Trading volume - January\n2026,NaN,January,2026
1,MY422033-W,Polyplastics Asia Pacific Sdn. Bhd.,Monthly - Trading volume - January\n2026,NaN,January,2026
2,JP4120001125937,Daicel Corporation,Monthly - Trading volume - January\n2026,NaN,January,2026
3,JP1010401053834,"Polyplastics Co.,Ltd.",Monthly - Trading volume - January\n2026,NaN,January,2026
4,KR1353110013606,"Kolon Industries, Inc.",Monthly - Trading volume - January\n2026,NaN,January,2026


In [7]:
df_volume_signals = (
    df_trading_long
    .groupby(["BvD ID number", "Company name Latin alphabet", "year"], dropna=False)["volume"]
    .agg(
        avg_vol="mean",
        std_vol="std",
        max_vol="max",
        min_vol="min",
    )
    .reset_index()
)

df_volume_signals["vol_stability_score"] = df_volume_signals["std_vol"] / df_volume_signals["avg_vol"]
df_volume_signals["vol_shock_ratio"] = df_volume_signals["max_vol"] / (df_volume_signals["min_vol"] + 1)

In [8]:
df_volume_trends = []

for (bvd_id, year), df_group in df_trading_long.groupby(["BvD ID number", "year"], dropna=False):
    df_group = df_group.sort_values("month")
    slope = relative_trend(df_group["volume"])

    df_volume_trends.append({
        "BvD ID number": bvd_id,
        "year": year,
        "vol_trend_slope": slope,
    })

df_volume_trends = pd.DataFrame(df_volume_trends)
df_volume_yearly = df_volume_signals.merge(
    df_volume_trends,
    on=["BvD ID number", "year"],
    how="left",
)

df_volume_yearly = add_join_year(df_volume_yearly, year_col="year")

print("Yearly trading shape:", df_volume_yearly.shape)
display(df_volume_yearly.head())

Yearly trading shape: (30304, 11)


,BvD ID number,Company name Latin alphabet,year,avg_vol,std_vol,max_vol,min_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,Join_Year
0,AE0001327927,Rever Events L.L.C,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012
1,AE0001327927,Rever Events L.L.C,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013
2,AE0001327927,Rever Events L.L.C,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014
3,AE0001327927,Rever Events L.L.C,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015
4,AE0001327927,Rever Events L.L.C,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016


## 2. Monthly market capitalization → yearly market cap signals

In [9]:
df_market_cap_raw = pd.read_csv(monthly_market_cap_path, sep=";", low_memory=False)

print("Shape:", df_market_cap_raw.shape)
display(df_market_cap_raw.head())

Shape: (1894, 387)


,Company name Latin alphabet,Risk level,BvD ID number,Monthly - Market Capitalisation - January\nth DKK Last avail. yr,Monthly - Market Capitalisation - January\nth DKK Year - 1,Monthly - Market Capitalisation - January\nth DKK Year - 2,Monthly - Market Capitalisation - January\nth DKK Year - 3,Monthly - Market Capitalisation - January\nth DKK Year - 4,Monthly - Market Capitalisation - January\nth DKK Year - 5,Monthly - Market Capitalisation - January\nth DKK Year - 6,...,Monthly - Market Capitalisation - December\nth DKK 2020,Monthly - Market Capitalisation - December\nth DKK 2019,Monthly - Market Capitalisation - December\nth DKK 2018,Monthly - Market Capitalisation - December\nth DKK 2017,Monthly - Market Capitalisation - December\nth DKK 2016,Monthly - Market Capitalisation - December\nth DKK 2015,Monthly - Market Capitalisation - December\nth DKK 2014,Monthly - Market Capitalisation - December\nth DKK 2013,Monthly - Market Capitalisation - December\nth DKK 2012,Monthly - Market Capitalisation - December\nth DKK 2011
0,Jeev Lifeworks LLP,Do not source,IN0038742908,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,MY422033-W,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
2,Daicel Corporation,Go ahead,JP4120001125937,15.822.500,15.510.954,19.010.998,13.764.740,13.407.109,13.696.649,20.354.184,...,13.334.310,21.303.160,23.281.440,24.667.305,27.258.712,37.522.834,26.257.920,16.057.724,13.529.696,12.653.192
3,"Polyplastics Co.,Ltd.",Take caution,JP1010401053834,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
4,"Kolon Industries, Inc.",Go ahead,KR1353110013606,6.566.939,3.417.462,5.455.712,6.556.883,9.265.650,6.101.344,6.346.316,...,6.166.066,7.739.139,9.013.737,13.160.487,10.869.171,9.189.113,6.758.854,7.037.893,8.436.620,7.904.583


In [10]:
market_cap_pattern = r"Market Capitalisation - (\w+)\n.*(20\d{2})"

df_market_cap_long = melt_monthly_source(
    df_source=df_market_cap_raw,
    id_vars=["BvD ID number", "Risk level"],
    pattern=market_cap_pattern,
    var_name="raw_header",
    value_name="value",
)

df_market_cap_long["BvD ID number"] = clean_id_like(df_market_cap_long["BvD ID number"])
df_market_cap_long["value"] = parse_numeric_with_thousands_dot(df_market_cap_long["value"])

print("Long shape:", df_market_cap_long.shape)
display(df_market_cap_long.head())

Long shape: (363648, 6)


,BvD ID number,Risk level,raw_header,value,month,year
0,IN0038742908,Do not source,Monthly - Market Capitalisation - January\nth ...,NaN,January,2026
1,MY422033-W,Go ahead,Monthly - Market Capitalisation - January\nth ...,NaN,January,2026
2,JP4120001125937,Go ahead,Monthly - Market Capitalisation - January\nth ...,15822500.0,January,2026
3,JP1010401053834,Take caution,Monthly - Market Capitalisation - January\nth ...,NaN,January,2026
4,KR1353110013606,Go ahead,Monthly - Market Capitalisation - January\nth ...,6566939.0,January,2026


In [11]:
df_market_cap_yearly = (
    df_market_cap_long
    .groupby(["BvD ID number", "Risk level", "year"], dropna=False)["value"]
    .agg(
        avg_market_cap="mean",
        market_cap_volatility=lambda x: x.std() / x.mean() if x.mean() not in [0, np.nan] else np.nan,
    )
    .reset_index()
)

df_market_cap_yearly = add_join_year(df_market_cap_yearly, year_col="year")

print("Yearly market cap shape:", df_market_cap_yearly.shape)
display(df_market_cap_yearly.head())

Yearly market cap shape: (30304, 6)


,BvD ID number,Risk level,year,avg_market_cap,market_cap_volatility,Join_Year
0,AE0001327927,Take caution,2011,NaN,NaN,2012
1,AE0001327927,Take caution,2012,NaN,NaN,2013
2,AE0001327927,Take caution,2013,NaN,NaN,2014
3,AE0001327927,Take caution,2014,NaN,NaN,2015
4,AE0001327927,Take caution,2015,NaN,NaN,2016


## 3. Beta / correlation / static market indicators

In [12]:
df_beta_raw = pd.read_csv(beta_path, sep=";", low_memory=False)

print("Shape:", df_beta_raw.shape)
display(df_beta_raw.head())

Shape: (1894, 79)


,Company name Latin alphabet,Risk level,Date - Current,Price trends - Last week\n%,Price trends - 4 weeks\n%,Price trends - 13 weeks\n%,Price trends - Quarter to date\n%,Price trends - Previous quarter to date\n%,Price trends - Year to date\n%,Price trends - 52 weeks\n%,...,Market price - Year to date - Low\nDKK,Market price - 52 week - High\nDKK,Market price - 52 week - Low\nDKK,Shares outstanding,Current market capitalisation\nth DKK,Date of last 12 months - EPS,Earnings per share\nDKK,Book value per share\nDKK,BvD ID number,BvD sectors
0,Jeev Lifeworks LLP,Do not source,NaN,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,NaN,n.a.,n.a.,IN0038742908,Business Services
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,NaN,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,NaN,n.a.,n.a.,MY422033-W,"Chemicals, Petroleum, Rubber & Plastic"
2,Daicel Corporation,Go ahead,04/03/2026,-11,-7,12,4,4,3,8,...,57,68,43,266.942.682,15.822.500,31/03/2025,7,55,JP4120001125937,"Chemicals, Petroleum, Rubber & Plastic"
3,"Polyplastics Co.,Ltd.",Take caution,NaN,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,NaN,n.a.,n.a.,JP1010401053834,"Chemicals, Petroleum, Rubber & Plastic"
4,"Kolon Industries, Inc.",Go ahead,04/03/2026,-18,1,24,24,32,31,62,...,177,310,114,27.519.091,6.650.519,31/12/2024,16,575,KR1353110013606,"Chemicals, Petroleum, Rubber & Plastic"


In [13]:
beta_columns_to_keep = [
    "BvD ID number",
    "BvD sectors",
    "Risk level",
    "Price trends - 52 weeks\n%",
    "Ref. index 1Beta1 year",
    "Earnings per share\nDKK",
    "Book value per share\nDKK",
    "Shares outstanding",
    "Current market capitalisation\nth DKK",
]

beta_columns_to_keep = [c for c in beta_columns_to_keep if c in df_beta_raw.columns]

df_beta = df_beta_raw[beta_columns_to_keep].copy()
df_beta["BvD ID number"] = clean_id_like(df_beta["BvD ID number"])

cols_with_thousands_dot = [
    c for c in [
        "Price trends - 52 weeks\n%",
        "Ref. index 1Beta1 year",
        "Earnings per share\nDKK",
        "Book value per share\nDKK",
    ] if c in df_beta.columns
]

for col in cols_with_thousands_dot:
    df_beta[col] = parse_numeric_with_thousands_dot(df_beta[col])

rename_map = {
    "BvD sectors": "supplier_sector",
    "Risk level": "moodys_risk_rating",
    "Ref. index 1Beta1 year": "market_beta_1y",
    "Current market capitalisation\nth DKK": "Current_market_capitalisation_DKK",
    "Book value per share\nDKK": "Book_value_per_share_DKK",
    "Price trends - 52 weeks\n%": "Price_trends_52_weeks_pct",
    "Earnings per share\nDKK": "Earnings_per_share_DKK",
}

rename_map = {k: v for k, v in rename_map.items() if k in df_beta.columns}
df_beta = df_beta.rename(columns=rename_map)

print("Static beta shape:", df_beta.shape)
display(df_beta.head())

Static beta shape: (1894, 9)


,BvD ID number,supplier_sector,moodys_risk_rating,Price_trends_52_weeks_pct,market_beta_1y,Earnings_per_share_DKK,Book_value_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK
0,IN0038742908,Business Services,Do not source,NaN,NaN,NaN,NaN,n.a.,n.a.
1,MY422033-W,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,NaN,NaN,NaN,NaN,n.a.,n.a.
2,JP4120001125937,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,8.0,1.0,7.0,55.0,266.942.682,15.822.500
3,JP1010401053834,"Chemicals, Petroleum, Rubber & Plastic",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
4,KR1353110013606,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,62.0,1.0,16.0,575.0,27.519.091,6.650.519


## 4. Monthly outstanding shares → yearly shares signal

In [14]:
df_shares_raw = pd.read_csv(monthly_shares_path, sep=";", low_memory=False)

print("Shape:", df_shares_raw.shape)
display(df_shares_raw.head())

Shape: (1894, 387)


,Company name Latin alphabet,Risk level_monthly_shares,BvD ID number,Monthly - Outstanding shares - January\nth Last avail. yr,Monthly - Outstanding shares - January\nth Year - 1,Monthly - Outstanding shares - January\nth Year - 2,Monthly - Outstanding shares - January\nth Year - 3,Monthly - Outstanding shares - January\nth Year - 4,Monthly - Outstanding shares - January\nth Year - 5,Monthly - Outstanding shares - January\nth Year - 6,...,Monthly - Outstanding shares - December\nth 2020,Monthly - Outstanding shares - December\nth 2019,Monthly - Outstanding shares - December\nth 2018,Monthly - Outstanding shares - December\nth 2017,Monthly - Outstanding shares - December\nth 2016,Monthly - Outstanding shares - December\nth 2015,Monthly - Outstanding shares - December\nth 2014,Monthly - Outstanding shares - December\nth 2013,Monthly - Outstanding shares - December\nth 2012,Monthly - Outstanding shares - December\nth 2011
0,Jeev Lifeworks LLP,Do not source,IN0038742908,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,MY422033-W,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
2,Daicel Corporation,Go ahead,JP4120001125937,"266942,682","276942,682","286942,682","302942,682","302942,682","302942,682","331942,682",...,"302942,682","331942,682","349942,682","349942,682","349942,682","364942,682","364942,682","364942,682","364942,682","364942,682"
3,"Polyplastics Co.,Ltd.",Take caution,JP1010401053834,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
4,"Kolon Industries, Inc.",Go ahead,KR1353110013606,"27519,091","27519,091","27519,091","27519,091","27519,091","26978,84","26978,84",...,"26978,84","26978,84","26978,84","25499,866","25151,402","25119,217","25103,951","25087,562","25055,838","25033,235"


In [15]:
shares_pattern = r"Monthly - Outstanding shares - (\w+)\n.*(20\d{2})"

df_shares_long = melt_monthly_source(
    df_source=df_shares_raw,
    id_vars=["BvD ID number", "Company name Latin alphabet", "Risk level_monthly_shares"],
    pattern=shares_pattern,
    var_name="raw_header",
    value_name="value",
)

df_shares_long["BvD ID number"] = clean_id_like(df_shares_long["BvD ID number"])
df_shares_long["value"] = parse_numeric_with_thousands_dot(df_shares_long["value"])

print("Long shape:", df_shares_long.shape)
display(df_shares_long.head())

Long shape: (363648, 7)


,BvD ID number,Company name Latin alphabet,Risk level_monthly_shares,raw_header,value,month,year
0,IN0038742908,Jeev Lifeworks LLP,Do not source,Monthly - Outstanding shares - January\nth 2026,NaN,January,2026
1,MY422033-W,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,Monthly - Outstanding shares - January\nth 2026,NaN,January,2026
2,JP4120001125937,Daicel Corporation,Go ahead,Monthly - Outstanding shares - January\nth 2026,NaN,January,2026
3,JP1010401053834,"Polyplastics Co.,Ltd.",Take caution,Monthly - Outstanding shares - January\nth 2026,NaN,January,2026
4,KR1353110013606,"Kolon Industries, Inc.",Go ahead,Monthly - Outstanding shares - January\nth 2026,NaN,January,2026


In [16]:
df_shares_yearly = (
    df_shares_long
    .groupby(
        ["BvD ID number", "Company name Latin alphabet", "Risk level_monthly_shares", "year"],
        dropna=False
    )["value"]
    .agg(avg_shares_outstanding="mean")
    .reset_index()
)

df_shares_yearly = add_join_year(df_shares_yearly, year_col="year")

print("Yearly shares shape:", df_shares_yearly.shape)
display(df_shares_yearly.head())

Yearly shares shape: (30304, 6)


,BvD ID number,Company name Latin alphabet,Risk level_monthly_shares,year,avg_shares_outstanding,Join_Year
0,AE0001327927,Rever Events L.L.C,Take caution,2011,NaN,2012
1,AE0001327927,Rever Events L.L.C,Take caution,2012,NaN,2013
2,AE0001327927,Rever Events L.L.C,Take caution,2013,NaN,2014
3,AE0001327927,Rever Events L.L.C,Take caution,2014,NaN,2015
4,AE0001327927,Rever Events L.L.C,Take caution,2015,NaN,2016


## 5. Monthly closing prices → yearly price signals

In [17]:
df_price_raw = pd.read_csv(monthly_closing_price_path, sep=";", low_memory=False)

print("Shape:", df_price_raw.shape)
display(df_price_raw.head())

Shape: (1894, 387)


,Company name Latin alphabet,Risk level_stock_closing_price,BvD ID number,Monthly - Closing price - January\nDKK Last avail. yr,Monthly - Closing price - January\nDKK Year - 1,Monthly - Closing price - January\nDKK Year - 2,Monthly - Closing price - January\nDKK Year - 3,Monthly - Closing price - January\nDKK Year - 4,Monthly - Closing price - January\nDKK Year - 5,Monthly - Closing price - January\nDKK Year - 6,...,Monthly - Closing price - December\nDKK 2020,Monthly - Closing price - December\nDKK 2019,Monthly - Closing price - December\nDKK 2018,Monthly - Closing price - December\nDKK 2017,Monthly - Closing price - December\nDKK 2016,Monthly - Closing price - December\nDKK 2015,Monthly - Closing price - December\nDKK 2014,Monthly - Closing price - December\nDKK 2013,Monthly - Closing price - December\nDKK 2012,Monthly - Closing price - December\nDKK 2011
0,Jeev Lifeworks LLP,Do not source,IN0038742908,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,MY422033-W,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
2,Daicel Corporation,Go ahead,JP4120001125937,"59,2730255","56,00781229","66,25364433","45,43677881","44,2562568","45,21201329","61,31837023",...,"44,01595118","64,17722542","66,52929381","70,48955907","77,89479158","102,8184322","71,95080554","44,00067517","37,07348081","34,67172288"
3,"Polyplastics Co.,Ltd.",Take caution,JP1010401053834,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
4,"Kolon Industries, Inc.",Go ahead,KR1353110013606,"238,6321357","124,1851322","198,2519124","238,2667075","336,6989996","226,1529321","235,2330992",...,"228,5519224","286,8595872","334,1039399","516,1002392","432,1496996","365,820044","269,2346805","280,5331504","336,7127564","315,7635493"


In [18]:
price_pattern = r"Monthly - Closing price - (\w+)\n.*(20\d{2})"

df_price_long = melt_monthly_source(
    df_source=df_price_raw,
    id_vars=["BvD ID number", "Company name Latin alphabet", "Risk level_stock_closing_price"],
    pattern=price_pattern,
    var_name="raw_header",
    value_name="value",
)

df_price_long["BvD ID number"] = clean_id_like(df_price_long["BvD ID number"])
df_price_long["value"] = parse_numeric_with_comma_decimal(df_price_long["value"])

print("Long shape:", df_price_long.shape)
display(df_price_long.head())

Long shape: (363648, 7)


,BvD ID number,Company name Latin alphabet,Risk level_stock_closing_price,raw_header,value,month,year
0,IN0038742908,Jeev Lifeworks LLP,Do not source,Monthly - Closing price - January\nDKK 2026,NaN,January,2026
1,MY422033-W,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,Monthly - Closing price - January\nDKK 2026,NaN,January,2026
2,JP4120001125937,Daicel Corporation,Go ahead,Monthly - Closing price - January\nDKK 2026,59.273026,January,2026
3,JP1010401053834,"Polyplastics Co.,Ltd.",Take caution,Monthly - Closing price - January\nDKK 2026,NaN,January,2026
4,KR1353110013606,"Kolon Industries, Inc.",Go ahead,Monthly - Closing price - January\nDKK 2026,238.632136,January,2026


In [19]:
df_price_yearly = (
    df_price_long
    .groupby(
        ["BvD ID number", "Company name Latin alphabet", "Risk level_stock_closing_price", "year"],
        dropna=False
    )["value"]
    .agg(
        avg_closing_price="mean",
        price_volatility_score=lambda x: x.std() / x.mean() if x.mean() not in [0, np.nan] else np.nan,
        price_trend_slope=relative_trend,
    )
    .reset_index()
)

df_price_yearly = add_join_year(df_price_yearly, year_col="year")

print("Yearly price shape:", df_price_yearly.shape)
display(df_price_yearly.head())

Yearly price shape: (30304, 8)


,BvD ID number,Company name Latin alphabet,Risk level_stock_closing_price,year,avg_closing_price,price_volatility_score,price_trend_slope,Join_Year
0,AE0001327927,Rever Events L.L.C,Take caution,2011,NaN,NaN,NaN,2012
1,AE0001327927,Rever Events L.L.C,Take caution,2012,NaN,NaN,NaN,2013
2,AE0001327927,Rever Events L.L.C,Take caution,2013,NaN,NaN,NaN,2014
3,AE0001327927,Rever Events L.L.C,Take caution,2014,NaN,NaN,NaN,2015
4,AE0001327927,Rever Events L.L.C,Take caution,2015,NaN,NaN,NaN,2016


## 6. Merge yearly stocks components into one view

The notebook stays focused on yearly stock-view construction.  
More interpretive feature engineering should happen later.

In [20]:
df_stock_view = (
    df_volume_yearly
    .merge(
        df_market_cap_yearly[[
            "BvD ID number",
            "year",
            "Join_Year",
            "avg_market_cap",
            "market_cap_volatility",
            "Risk level",
        ]],
        on=["BvD ID number", "year", "Join_Year"],
        how="left",
    )
    .merge(
        df_beta,
        on=["BvD ID number"],
        how="left",
    )
    .merge(
        df_shares_yearly[[
            "BvD ID number",
            "year",
            "Join_Year",
            "avg_shares_outstanding",
            "Risk level_monthly_shares",
        ]],
        on=["BvD ID number", "year", "Join_Year"],
        how="left",
    )
    .merge(
        df_price_yearly[[
            "BvD ID number",
            "year",
            "Join_Year",
            "avg_closing_price",
            "price_volatility_score",
            "price_trend_slope",
            "Risk level_stock_closing_price",
        ]],
        on=["BvD ID number", "year", "Join_Year"],
        how="left",
    )
)

print("Stock view shape:", df_stock_view.shape)
display(df_stock_view.head())

Stock view shape: (30304, 28)


,BvD ID number,Company name Latin alphabet,year,avg_vol,std_vol,max_vol,min_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,...,Earnings_per_share_DKK,Book_value_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK,avg_shares_outstanding,Risk level_monthly_shares,avg_closing_price,price_volatility_score,price_trend_slope,Risk level_stock_closing_price
0,AE0001327927,Rever Events L.L.C,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,n.a.,n.a.,NaN,Take caution,NaN,NaN,NaN,Take caution
1,AE0001327927,Rever Events L.L.C,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,n.a.,n.a.,NaN,Take caution,NaN,NaN,NaN,Take caution
2,AE0001327927,Rever Events L.L.C,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,n.a.,n.a.,NaN,Take caution,NaN,NaN,NaN,Take caution
3,AE0001327927,Rever Events L.L.C,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,n.a.,n.a.,NaN,Take caution,NaN,NaN,NaN,Take caution
4,AE0001327927,Rever Events L.L.C,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,n.a.,n.a.,NaN,Take caution,NaN,NaN,NaN,Take caution


## Sanity checks

In [21]:
print("Rows:", len(df_stock_view))
print("Unique BvD IDs:", df_stock_view["BvD ID number"].nunique())
print("Duplicate BvD-year-join_year rows:", df_stock_view.duplicated(["BvD ID number", "year", "Join_Year"]).sum())

display(
    df_stock_view[[
        "BvD ID number",
        "year",
        "Join_Year",
        "avg_vol",
        "vol_stability_score",
        "vol_shock_ratio",
        "vol_trend_slope",
        "avg_market_cap",
        "market_cap_volatility",
        "market_beta_1y",
        "Earnings_per_share_DKK",
        "avg_closing_price",
        "price_volatility_score",
        "price_trend_slope",
    ]].head(20)
)

Rows: 30304
Unique BvD IDs: 1893
Duplicate BvD-year-join_year rows: 0


,BvD ID number,year,Join_Year,avg_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,avg_market_cap,market_cap_volatility,market_beta_1y,Earnings_per_share_DKK,avg_closing_price,price_volatility_score,price_trend_slope
0,AE0001327927,2011,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AE0001327927,2012,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AE0001327927,2013,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AE0001327927,2014,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AE0001327927,2015,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AE0001327927,2016,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AE0001327927,2017,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,AE0001327927,2018,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,AE0001327927,2019,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,AE0001327927,2020,2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
key_output_cols = [
    "avg_vol",
    "std_vol",
    "max_vol",
    "min_vol",
    "vol_stability_score",
    "vol_shock_ratio",
    "vol_trend_slope",
    "avg_market_cap",
    "market_cap_volatility",
    "market_beta_1y",
    "Earnings_per_share_DKK",
    "Book_value_per_share_DKK",
    "avg_shares_outstanding",
    "avg_closing_price",
    "price_volatility_score",
    "price_trend_slope",
]

key_output_cols = [c for c in key_output_cols if c in df_stock_view.columns]

df_output_missingness = pd.DataFrame({
    "feature": key_output_cols,
    "missing_pct": [df_stock_view[c].isna().mean() * 100 for c in key_output_cols],
    "n_unique": [df_stock_view[c].nunique(dropna=True) for c in key_output_cols],
}).sort_values("missing_pct", ascending=False)

display(df_output_missingness)

,feature,missing_pct,n_unique
12,avg_shares_outstanding,99.194826,123
4,vol_stability_score,99.158527,253
1,std_vol,99.155227,254
6,vol_trend_slope,99.155227,254
0,avg_vol,99.059530,280
2,max_vol,99.059530,274
3,min_vol,99.059530,271
5,vol_shock_ratio,99.059530,274
8,market_cap_volatility,96.502112,1060
14,price_volatility_score,96.502112,1060


## Save final stock view

In [23]:
df_stock_view.to_csv(output_path, index=False)

print("Saved file to:", output_path)
print("Shape:", df_stock_view.shape)
print("Unique BvD IDs:", df_stock_view["BvD ID number"].nunique())

Saved file to: /Users/Thomas/Desktop/Master Thesis/Data/processed/stock_view.csv
Shape: (30304, 28)
Unique BvD IDs: 1893
